# Line Identification

[Chapter 5](05-arc-lamps.ipynb) introduced the arc frames; Chapter 6 straightened them. What
remains is to turn the emission lines they carry into (pixel, wavelength) pairs, the raw
material every wavelength solution is fitted to.

The work divides into two parts.

- **Detection**: locating the emission lines and measuring their centroids to sub-pixel
  precision. `specreduce.line_matching.find_arc_lines` does this. In the normal workflow it is
  not called directly: `WavelengthCalibration1D` runs it on the arc spectra it is given, through
  its `find_lines` method, as Chapter 10 shows. This chapter runs it by hand to look at what it
  returns. Nothing downstream is tied to it either way, because the calibration also accepts
  observed line positions measured with another detection routine, which serve it equally
  well.
- **Identification**: deciding which wavelength each detected line has. In the interactive
  workflow this is a human step. The detected lines, labeled with their pixel positions, are
  compared against reference figures of the same lamps through the same instrument setup, on
  which the lines are labeled with their wavelengths, and a handful of strong lines present in
  both are paired up.

This chapter does the detection, introduces the reference material, and walks through the
identification. It ends with a set of anchor lines that
[Chapter 10](10-wavelength-solution-interactive.ipynb) fits a solution to and then grows.
[Chapter 11](11-wavelength-solution-automated.ipynb) needs no anchors, because it starts from
prior knowledge of the solution.

In [ ]:
import sys
import warnings

sys.path.append("../src")

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from specreduce.calibration_data import (
    get_available_line_catalogs,
    load_pypeit_calibration_lines,
)
from specreduce.extract import BoxcarExtract
from specreduce.line_matching import find_arc_lines
from specreduce.tilt_solution import TiltSolution
from specreduce.tracing import FlatTrace

from common import (
    ANCHOR_PIXELS,
    ANCHOR_WAVELENGTHS,
    GTC_HGAR,
    TILT_SOLUTION_FILE,
    fit_tilt_solution,
    read_data,
)

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
nx = obj.data.shape[1]

## Where the previous chapters left off

The setup cell reads Chapter 6's tilt solution, rectifies the arcs, and collapses each to 1D with
a 15-row `BoxcarExtract` (Chapter 8) at the reference row.

In [ ]:
reference = FlatTrace(arcs[0], 135)

if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

rectified_arcs = [solution.resample(frame) for frame in arcs]


spectra = [
    BoxcarExtract(frame, reference, width=15).spectrum for frame in rectified_arcs
]

fig, axs = plt.subplots(3, figsize=(8, 5.0), sharex="all", constrained_layout=True)

for ax, lamp, spectrum in zip(axs, lamps, spectra):
    ax.plot(spectrum.flux.value, lw=0.6)
    ax.text(0.01, 0.9, lamp, transform=ax.transAxes, va="top", size=8)
    ax.set(yscale="log", ylim=(30, 1e7), ylabel="Flux [e$^-$]")

axs[-1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]");

The lines sit on a floor that is neither zero nor flat: the lamp continuum falls toward the red on
HgAr and rises by an order of magnitude under the dense part of the Ne spectrum. `find_arc_lines`
subtracts a sigma-clipped baseline before thresholding, as `TiltCorrection.find_arc_lines` did in
Chapter 6, so the extractions go in as they are. It accepts any `astropy.nddata` uncertainty type.
Given none, it estimates a single noise level with the DER_SNR estimator of
[Stoehr et al. (2008)](https://ui.adsabs.harvard.edu/abs/2008ASPC..394..505S/abstract), which on
these line-rich arcs runs two to six times above the propagated uncertainty, so pass the
propagated uncertainty when you have it.

The 15-row aperture is wider than the raw frames would allow, because a tilted line summed over
many rows smears across columns and blunts its centroid. On the rectified frames the lines are
vertical and the width is set by signal-to-noise alone; on an instrument with little tilt a wide
aperture is safe from the start.

## Finding the lines

`find_arc_lines` subtracts the baseline, calls `find_lines_threshold` to locate peaks rising above
`noise_factor` times the uncertainty, and fits a Gaussian within `window * fwhm` pixels of each
detection to refine its center. It returns a `QTable`:

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ne_lines = find_arc_lines(spectra[1], fwhm=2.5, noise_factor=15)

print(f"columns: {ne_lines.colnames}")
print(ne_lines[:5])

The `centroid` is what wavelength calibration consumes; the fitted `fwhm` and the `amplitude` turn
out to be the key to quality control.

`noise_factor` is the detection threshold in units of the uncertainty, so it has to be re-tuned
after any step that changes the noise model, and neither direction is free. Lowering it lets
adjacent lines merge into one region whose Gaussian refit runs away, and the width filter below
then discards it: at `noise_factor=10` HgAr loses its bright line at pixel 550 this way. Raising
it drops the faintest lines. We use **15**. `fwhm` does not change how many lines are found; it
sizes the refit window, so set it to the instrumental line width, the 2.5 pixels measured in
Chapter 5.

Not everything the threshold flags is a line. Noise spikes, the flanks of bright lines and blended
pairs all get through, and the refit exposes them, because a genuine arc line comes back near the
instrumental width. Keeping detections whose fitted width lies between 1.5 and 4 pixels removes
them: narrower are spikes and flanks, wider are failed refits spread across a broad feature, the
worst here over 200 pixels. The cut drops 7 of 28 detections on HgAr, 6 of 28 on Ne and 3 of 29 on
Xe. The failed refits are also what the `astropy` convergence warnings complain about, so the
warnings are silenced.

What survives, drawn on the three spectra:

In [ ]:
detected = {}
for lamp, spectrum in zip(lamps, spectra):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        table = find_arc_lines(spectrum, fwhm=2.5, noise_factor=15)
    widths = np.asarray(table["fwhm"])
    detected[lamp] = table[(widths > 1.5) & (widths < 4.0)]

total = sum(len(detected[lamp]) for lamp in lamps)
print(f"detected across three lamps: {total} lines")

fig, axs = plt.subplots(3, figsize=(8, 5.0), sharex="all", constrained_layout=True)

for ax, lamp, spectrum in zip(axs, lamps, spectra):
    ax.plot(spectrum.flux.value, lw=0.6)
    for centroid in np.asarray(detected[lamp]["centroid"]):
        ax.axvline(centroid, c="C1", lw=0.5, alpha=0.7)
    ax.text(
        0.01,
        0.9,
        f"{lamp}  ({len(detected[lamp])} lines)",
        transform=ax.transAxes,
        va="top",
        size=8,
    )
    ax.set(yscale="log", ylim=(30, 1e7), ylabel="Flux [e$^-$]")

axs[-1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]");

HgAr and Xe show a marker on nearly every bright, well-separated line, plus a run of faint lines at
the red end of Xe. Ne is different: its dense comb of bright lines between pixels 160 and 510
carries a single marker, near pixel 418, because `find_lines_threshold` treats each contiguous run
of pixels above the threshold as one line, and in that comb every pixel clears it. The same
happens to Xe's bright group between pixels 610 and 830, which yields only the line near 749, and
to the close pairs in HgAr. The detections are therefore the **isolated** lines, 69 across the
three lamps, which is what a wavelength fit wants: an unresolved group could not be identified
unambiguously anyway. The axis is still pixels.

## Reference line lists

The wavelengths come from a catalog, and `specreduce` ships accessors for the
[`pypeit`](https://pypeit.readthedocs.io) line lists.

In [ ]:
catalogs = get_available_line_catalogs()
available = sorted(catalogs["pypeit"])
print(f"{len(available)} pypeit line lists available, including:")
print(
    "  "
    + ", ".join(
        n
        for n in available
        if n in ("ArI", "HgI", "NeI", "XeI", "CdI", "HeI", "ThAr", "OH_GMOS")
    )
)

The names are *ions*, not lamps, so an HgAr lamp needs `HgI` and `ArI` together. The `OH_*` entries
are sky airglow lists, for calibrating on the sky's own lines.

:::{note}
`load_pypeit_calibration_lines` **downloads** the lists on first use and caches them locally, so
this cell needs network access the first time it runs.
:::

In [ ]:
lamp_ions = {"HgAr": ["HgI", "ArI"], "Ne": ["NeI"], "Xe": ["XeI"]}
grism_range = (5000, 10500) * u.AA

catalog = {}
for lamp, ions in lamp_ions.items():
    lines = load_pypeit_calibration_lines(ions)["wavelength"].to(u.AA)
    inside = (lines > grism_range[0]) & (lines < grism_range[1])
    catalog[lamp] = lines[inside]
    print(
        f"{lamp:>5}: {len(lines):4d} catalog lines, {len(catalog[lamp]):4d} within "
        f"{grism_range[0].value:.0f}-{grism_range[1].value:.0f} A, "
        f"{len(detected[lamp]):3d} detected"
    )

The numbers mismatch in both directions. HgAr has far more catalog lines in range than detections,
because most entries are too faint to appear in a short exposure; Xe has more detections than
catalog entries, because the `XeI` list is thin at these wavelengths. Which detection goes with
which catalog line has to be worked out, knowing that many on both sides have no partner.

### Instrument-specific line lists

An observatory usually publishes something more useful than a generic catalog: a list per lamp and
grism of the lines its instrument actually shows. GTC does this for OSIRIS, and the HgAr list for
R1000R is short enough to keep in `common.py`:

In [ ]:
print(f"GTC HgAr line list for R1000R: {len(GTC_HGAR)} lines (air wavelengths)")
print(GTC_HGAR)
print(f"\npypeit HgI + ArI lines in the same range: {len(catalog['HgAr'])}")

Eleven lines against 142: a list of the lines that appear is worth more than a list of those that
could. The GTC wavelengths are in air and the `pypeit` lists in vacuum, which matters when
Chapter 10 combines them.

## Identifying lines by hand

Pairing detections with catalog wavelengths is done by a person with a **reference figure**, a plot
of the same lamp through the same instrument and grism on a wavelength axis, with the lines
labeled. Observatories publish them for their configurations, lamp manufacturers publish lamp
spectra, and arc atlases from similar instruments serve when nothing closer exists. For OSIRIS,
GTC's [arc line maps](https://www.gtc.iac.es/instruments/osiris/osiris.php) include one figure per lamp and grism, and the R1000R maps
for [HgAr](https://www.gtc.iac.es/instruments/osiris/media/R1000R-HgAr.jpg), [Ne](https://www.gtc.iac.es/instruments/osiris/media/R1000R-Ne.jpg) and
[Xe](https://www.gtc.iac.es/instruments/osiris/media/R1000R-Xe.jpg) are the references for this dataset.

The HgAr map is easy to read. The Hg green line at 5460.735 Å dominates, alone at the blue end,
followed by a close Hg pair at 5769.598 and 5790.663 Å. A cluster of Ar lines between 6965 and
8521 Å holds two bright pairs, 7635.106 with 7724.207 Å and 8115.311 with 8264.522 Å, and the
strongest of the isolated red lines is 9122.967 Å.

Below, the detected lines are labeled by pixel, and the lower panel stands in for the reference
figure: the GTC list on a wavelength axis, which is what a catalog gives when no figure exists.

In [ ]:
hgar = detected["HgAr"]
centroids = np.asarray(hgar["centroid"])
flux = spectra[0].flux.value

fig, axs = plt.subplots(
    2, figsize=(8, 5.4), height_ratios=[3, 1], constrained_layout=True
)

ax = axs[0]
ax.plot(flux, lw=0.6, c="0.2")
for centroid in centroids:
    ax.axvline(centroid, c="C1", lw=0.5, alpha=0.6)
    ax.text(
        centroid,
        1.8 * flux[int(round(centroid))],
        f"{centroid:.0f}",
        rotation=90,
        ha="center",
        va="bottom",
        size=6,
    )
ax.set(
    yscale="log",
    xlim=(0, nx - 1),
    ylim=(300, 3e7),
    xlabel="Dispersion [pix]",
    ylabel="Flux [e$^-$]",
    title="HgAr, detected lines labeled by pixel",
)

ax = axs[1]
ax.vlines(GTC_HGAR, 0, 1, color="C0", lw=1.0)
for i, wavelength in enumerate(GTC_HGAR):
    # Nudge the labels of lines closer than 50 A apart so they do not overlap.
    close_below = i > 0 and wavelength - GTC_HGAR[i - 1] < 50
    close_above = i < len(GTC_HGAR) - 1 and GTC_HGAR[i + 1] - wavelength < 50
    nudge = 45 if close_below else (-45 if close_above else 0)
    ax.text(
        wavelength + nudge,
        1.08,
        f"{wavelength:.3f}",
        rotation=90,
        ha="center",
        va="bottom",
        size=6,
    )
ax.set(
    xlim=(5000, 10500),
    ylim=(0, 3.2),
    yticks=[],
    xlabel=r"Wavelength [$\AA$]",
    title="GTC HgAr line list for R1000R",
)
for ax in axs:
    ax.title.set_size(8)

The real map also shows the brightnesses, which make the identification quick:

- The brightest detection by far, near pixel 87 with nothing else within 300 pixels, can only be
  the green line at 5460.735 Å.
- The brightest lines mid-frame, near pixels 533 and 550, are a close pair with the first the
  stronger: the Ar pair 7635.106 and 7724.207 Å. Their separation, 17 pixels for 89 Å, gives a
  first dispersion that every further pairing must match.
- The next bright pair, near pixels 622 and 650, is 8115.311 and 8264.522 Å, and the isolated line
  near pixel 803 is 9122.967 Å, both where that dispersion puts them.

Take lines that are **strong and isolated in both** the spectrum and the reference, and match the
**pattern** of spacings and brightnesses rather than single lines. Leave out the rest. The Hg pair
at 5769.598 and 5790.663 Å is only four pixels apart here and blended, with no detection surviving
the width filter, and the fainter Ar lines between the bright pairs are easier to confuse with a
neighbor; Chapter 10's refinement recovers them once a solution exists.

The six pairs are stored in `common.py` as `ANCHOR_PIXELS` and `ANCHOR_WAVELENGTHS`, with the
pixels rounded to whole numbers, the precision a plot can be read to; Chapter 10 snaps them to the
measured centroids. Here they are at the detections they snap to:

In [ ]:
nearest = centroids[np.abs(centroids[:, None] - ANCHOR_PIXELS[None, :]).argmin(axis=0)]

fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)
ax.plot(flux, lw=0.6, c="0.2")
for centroid, wavelength in zip(nearest, ANCHOR_WAVELENGTHS):
    ax.axvline(centroid, c="C3", lw=0.8)
    ax.text(
        centroid,
        1.8 * flux[int(round(centroid))],
        f"{wavelength:.3f}",
        rotation=90,
        ha="center",
        va="bottom",
        size=6,
    )
ax.set(
    yscale="log",
    xlim=(0, nx - 1),
    ylim=(300, 3e7),
    xlabel="Dispersion [pix]",
    ylabel="Flux [e$^-$]",
);

A quick check, not a step of the reduction, shows that the six pairs agree with each other: a
quadratic through them predicts where every other line on the GTC list should fall.

In [ ]:
quadratic = np.polyfit(nearest, ANCHOR_WAVELENGTHS, 2)
grid = np.arange(nx)
predicted = np.interp(GTC_HGAR, np.polyval(quadratic, grid), grid)

print(
    f"{'wavelength [A]':>14}  {'predicted pixel':>15}  {'nearest detection':>17}  "
    f"{'offset [pix]':>12}"
)
for wavelength, pixel in zip(GTC_HGAR, predicted):
    offset = centroids - pixel
    j = np.abs(offset).argmin()
    tag = "anchor" if wavelength in ANCHOR_WAVELENGTHS else ""
    print(
        f"{wavelength:>14.3f}  {pixel:>15.1f}  {centroids[j]:>17.2f}  {offset[j]:>12.1f}  {tag}"
    )

Every line the width filter kept lands within 2 pixels of its prediction, the three Ar lines that
were not anchors included, and the blended Hg pair is predicted at pixels 155 and 160, where no
detection survived. The identification is self-consistent, and the fit with all the lines is
Chapter 10's.

## What this means for the next chapters

`find_arc_lines` returns a sub-pixel centroid for every isolated line, and its fitted width and
amplitude tell a real line from a spurious one. From here on `WavelengthCalibration1D` runs it
through `find_lines`, and the matching tolerance keeps spurious detections out of the fit;
positions from another detection routine can go into the same class instead.

The six anchors are certainly right but do not make a solution. A polynomial through them predicts
where every catalog line falls, the detections that land close enough are matched, and the fit is
redone with all of them at a tighter tolerance. `WavelengthCalibration1D` does that in two calls,
which [Chapter 10](10-wavelength-solution-interactive.ipynb) runs one at a time.
[Chapter 11](11-wavelength-solution-automated.ipynb) takes a different route, for a pipeline on a
stable instrument that already knows the solution to within a small range: `fit_dispersion`
searches inside that range, and no anchors are needed.

## Summary

- **Detection and identification are different problems.** `find_arc_lines` handles detection,
  normally through `WavelengthCalibration1D.find_lines`; identification is where reductions go
  wrong.
- **Rectify first if the tilt calls for it.** On a strongly tilted frame a wide extraction aperture
  smears the lines; once the frame is rectified, the aperture width is set by signal-to-noise
  alone.
- **`noise_factor` is the detection threshold in units of the uncertainty.** Re-tune it when the
  noise model changes: too low and bright lines merge with their neighbors and are lost, too high
  and the faint ones drop out. `fwhm` only sizes the refit window.
- **The fitted width tells a line from a spurious detection**, and unresolved groups collapse to
  one detection, so the detector returns the isolated lines.
- **Identify against a reference figure.** Choose lines that are strong and isolated in both the
  spectrum and the reference, match the pattern of spacings and brightnesses, and leave blends and
  faint lines to the refinement.
- **Use an instrument-specific line list when one exists.** GTC's list holds 11 HgAr lines where
  the generic catalogs hold 142. Mind air against vacuum wavelengths when lists are combined.

The six anchors are stored in `common.py` as `ANCHOR_PIXELS` and `ANCHOR_WAVELENGTHS`, and
Chapter 10 imports them.

Next: [Interactive Wavelength Solution](10-wavelength-solution-interactive.ipynb).